# Epic Renewal Data-Entry Agent (prototype)

Goal: read a renewal document (PDF, possibly scanned), extract the fields a broker would otherwise retype, and push them into Applied Epic via its API — with a human approval step before anything is actually written.

This is a **prototype using the same kernel/venv as the rest of `lca-lc-foundations`**. The Applied Epic client below is a **stub**: no API credentials are wired up yet, so it returns fake data and prints the payload it *would* send. Swap `EpicClient` for real HTTP calls once you have sandbox access from Applied Systems — everything upstream of it (extraction, mapping, approval flow) doesn't need to change.

Pipeline: `PDF -> text/vision extraction -> structured RenewalSubmission -> field mapping -> Epic payload -> human approval -> EpicClient.update_policy()`

**Before running:** `uv add httpx pymupdf` (httpx for the future Epic REST client, pymupdf/fitz to rasterize scanned pages for vision-model extraction).

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. Target schema

This is the data contract between "what we pulled off the renewal doc" and "what Epic needs." Keep it small and specific to the fields you're actually automating first — named insured, policy identifiers, term dates, and premium are the highest-value, lowest-risk fields to start with. Expand it (coverages, limits, forms, additional insureds) once the pipeline is trustworthy for the basics.

Field names/types should ultimately mirror whatever Epic's API expects for a policy/renewal record — update this once you have real API docs.

In [3]:
from datetime import date
from typing import Optional
from pydantic import BaseModel, Field


class Coverage(BaseModel):
    coverage_type: str = Field(description="e.g. 'General Liability', 'Commercial Property'")
    limit: Optional[str] = Field(default=None, description="e.g. '$1,000,000 per occurrence'")
    deductible: Optional[str] = None


class RenewalSubmission(BaseModel):
    """Fields extracted from a renewal document, destined for Applied Epic."""

    named_insured: str
    mailing_address: Optional[str] = None
    policy_number: Optional[str] = Field(default=None, description="Expiring policy number, if shown")
    carrier: Optional[str] = Field(default=None, description="Issuing carrier / company name")
    line_of_business: Optional[str] = Field(default=None, description="e.g. 'Commercial Auto', 'BOP'")
    effective_date: Optional[date] = None
    expiration_date: Optional[date] = None
    total_premium: Optional[str] = Field(default=None, description="As shown on the doc, e.g. '$4,250.00'")
    coverages: list[Coverage] = Field(default_factory=list)
    extraction_notes: Optional[str] = Field(
        default=None,
        description="Anything ambiguous, illegible, or missing that a human should double check",
    )

## 2. Ingest the renewal document

Two paths depending on the document:
- **Text-based PDF** (digitally generated, e.g. carrier renewal packets): `pypdf` pulls selectable text directly — cheap and reliable.
- **Scanned/image PDF** (no selectable text): render the pages to images with `pymupdf` and hand them to a vision-capable model instead.

`load_renewal_document` tries text extraction first and falls back to page images when the yield is too thin to be real text (a scanned page usually yields near-zero characters).

In [4]:
import base64
from pathlib import Path

from pypdf import PdfReader
import pymupdf

MIN_CHARS_PER_PAGE = 40  # below this, assume the page is a scan with no real text layer


def _extract_text(pdf_path: Path) -> tuple[str, bool]:
    """Returns (text, looks_text_native). looks_text_native is False if extraction yielded too little to trust."""
    reader = PdfReader(str(pdf_path))
    pages_text = [page.extract_text() or "" for page in reader.pages]
    text = "\n\n".join(pages_text)
    avg_chars = len(text) / max(len(pages_text), 1)
    return text, avg_chars >= MIN_CHARS_PER_PAGE


def _render_pages_as_data_urls(pdf_path: Path, dpi: int = 200) -> list[str]:
    """Rasterize each page to a PNG data URL for a vision-capable model."""
    doc = pymupdf.open(str(pdf_path))
    zoom = dpi / 72
    urls = []
    for page in doc:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
        b64 = base64.b64encode(pix.tobytes("png")).decode("utf-8")
        urls.append(f"data:image/png;base64,{b64}")
    doc.close()
    return urls


def load_renewal_document(pdf_path: str) -> dict:
    """Returns {'mode': 'text', 'text': ...} or {'mode': 'vision', 'image_urls': [...]}."""
    path = Path(pdf_path)
    text, looks_text_native = _extract_text(path)
    if looks_text_native:
        return {"mode": "text", "text": text}
    return {"mode": "vision", "image_urls": _render_pages_as_data_urls(path)}

## 3. Structured extraction

Same model either way — the only difference is whether the human message carries text or image blocks. Use a stronger model here; getting policy numbers and dates right matters more than speed/cost for this step.

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

extraction_model = init_chat_model("claude-sonnet-4-5").with_structured_output(RenewalSubmission)

EXTRACTION_PROMPT = (
    "You are reviewing an insurance renewal document for a retail brokerage. "
    "Extract the fields into the given schema exactly as they appear on the document. "
    "Do not guess or fill in values that aren't present — leave them null and note the gap in "
    "extraction_notes instead. Flag anything ambiguous, cut off, or hard to read."
)


def extract_renewal_data(pdf_path: str) -> RenewalSubmission:
    doc = load_renewal_document(pdf_path)
    if doc["mode"] == "text":
        content = f"{EXTRACTION_PROMPT}\n\n---DOCUMENT TEXT---\n{doc['text']}"
    else:
        content = [{"type": "text", "text": EXTRACTION_PROMPT}] + [
            {"type": "image_url", "image_url": {"url": url}} for url in doc["image_urls"]
        ]
    return extraction_model.invoke([HumanMessage(content=content)])

## 4. Applied Epic client (stub)

**Placeholder only** — not wired to a real endpoint. Applied Epic's API (via Applied Systems' partner/Epic Exchange program) is OAuth2 client-credentials style: you'll get a token URL, an API base URL, and agency-scoped credentials once you have an API agreement in place. Fill in `EPIC_API_BASE_URL`, `EPIC_CLIENT_ID`, `EPIC_CLIENT_SECRET` in `.env` and replace the method bodies below with real `httpx` calls when you have docs — the method signatures (`find_policy`, `update_policy`) are what the rest of this notebook depends on, so keep those stable.

Until then, this stub just logs what it *would* send, so you can validate the extraction → mapping pipeline end to end without touching a live BMS.

In [6]:
import os


class EpicClient:
    """Stub client. Swap internals for real Applied Epic REST calls once you have API access."""

    def __init__(self):
        self.base_url = os.getenv("EPIC_API_BASE_URL", "<not configured>")
        self.client_id = os.getenv("EPIC_CLIENT_ID")
        self.connected = bool(self.client_id)

    def find_policy(self, named_insured: str, policy_number: Optional[str] = None) -> dict:
        """TODO: GET /policies?search=... Returns a fake match for now."""
        print(f"[EpicClient STUB] would search Epic for named_insured={named_insured!r} policy_number={policy_number!r}")
        return {"epic_policy_id": "STUB-POLICY-ID", "named_insured": named_insured}

    def update_policy(self, epic_policy_id: str, payload: dict) -> dict:
        """TODO: PATCH/PUT the renewal fields onto the matched Epic policy record."""
        print(f"[EpicClient STUB] would update Epic policy {epic_policy_id} with:")
        for k, v in payload.items():
            print(f"    {k}: {v}")
        return {"status": "stubbed", "epic_policy_id": epic_policy_id}


epic_client = EpicClient()

## 5. Map extracted fields → Epic payload

Keep this mapping explicit and separate from extraction. When you get real field names from Epic's API docs, this is the only function that should need to change.

In [7]:
def to_epic_payload(data: RenewalSubmission) -> dict:
    return {
        "InsuredName": data.named_insured,
        "MailingAddress": data.mailing_address,
        "PolicyNumber": data.policy_number,
        "Carrier": data.carrier,
        "LOB": data.line_of_business,
        "EffectiveDate": data.effective_date.isoformat() if data.effective_date else None,
        "ExpirationDate": data.expiration_date.isoformat() if data.expiration_date else None,
        "TotalPremium": data.total_premium,
        "Coverages": [c.model_dump() for c in data.coverages],
    }

## 6. Agent with a human-approval gate

Writing into a live BMS is not something to let an LLM do unsupervised, especially before the extraction step has a track record. `extract_and_review` runs automatically; `apply_to_epic` requires human sign-off via `HumanInTheLoopMiddleware`, same pattern as the email agent in module 3.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool


@tool
def extract_and_review(pdf_path: str) -> dict:
    """Extract renewal fields from a document and return them for review, without writing anything."""
    data = extract_renewal_data(pdf_path)
    return {"extracted": data.model_dump(mode="json"), "epic_payload_preview": to_epic_payload(data)}


@tool
def apply_to_epic(pdf_path: str) -> dict:
    """Extract renewal fields and write them into the matching Applied Epic policy record."""
    data = extract_renewal_data(pdf_path)
    match = epic_client.find_policy(data.named_insured, data.policy_number)
    return epic_client.update_policy(match["epic_policy_id"], to_epic_payload(data))


agent = create_agent(
    model="gpt-5-nano",
    tools=[extract_and_review, apply_to_epic],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "extract_and_review": False,
                "apply_to_epic": True,
            },
            description_prefix="Writing to Applied Epic requires approval",
        ),
    ],
)

## 7. Try it

`resources/sample_renewal.pdf` is a fictional, text-native renewal letter (fake insured, carrier, policy number, amounts) for a Businessowners Policy — good for a first end-to-end run of the text-extraction path. `extract_renewal_data` alone is the fastest way to sanity-check extraction quality before looping in the agent/Epic layer.

To test the scanned/vision fallback path, print `sample_renewal.pdf` and re-scan it (or otherwise flatten it to an image-only PDF) so it has no selectable text layer, and point `sample_pdf` at that instead.

In [9]:
sample_pdf = "resources/sample_renewal.pdf"
result = extract_renewal_data(sample_pdf)

print(result.model_dump_json(indent=2))
print("\n--- would map to this Epic payload ---")
print(to_epic_payload(result))

{
  "named_insured": "Blue Ridge Cabinetry, LLC",
  "mailing_address": "482 Millwright Lane, Asheville, NC 28801",
  "policy_number": "BOP-4471829-03",
  "carrier": "Acme Mutual Insurance Company",
  "line_of_business": "Businessowners Policy (BOP)",
  "effective_date": "2026-09-15",
  "expiration_date": "2027-09-15",
  "total_premium": "$6,842.00",
  "coverages": [
    {
      "coverage_type": "General Liability",
      "limit": "$1,000,000 per occurrence / $2,000,000 aggregate",
      "deductible": "$1,000"
    },
    {
      "coverage_type": "Commercial Property",
      "limit": "$850,000 building",
      "deductible": "$2,500"
    },
    {
      "coverage_type": "Business Personal Property",
      "limit": "$150,000",
      "deductible": "$2,500"
    },
    {
      "coverage_type": "Business Income",
      "limit": "$250,000 / 12 months",
      "deductible": "N/A"
    }
  ],
  "extraction_notes": null
}

--- would map to this Epic payload ---
{'InsuredName': 'Blue Ridge Cabinetry, 

## Next steps before this touches real data

1. **Get Epic API access.** Contact Applied Systems (or your Applied Epic account rep) about their partner/API program to get a sandbox, OAuth credentials, and real endpoint docs — then fill in `.env` (`EPIC_API_BASE_URL`, `EPIC_CLIENT_ID`, `EPIC_CLIENT_SECRET`) and replace `EpicClient`'s stub methods.
2. **Validate extraction accuracy on real (or de-identified) sample docs** across your actual carriers/lines of business before trusting it on live renewals — accuracy will vary a lot by carrier form layout.
3. **Confirm the Epic match** (`find_policy`) is unambiguous before allowing an update — wrong-policy writes are the costly failure mode, not wrong-field writes.
4. **Log every extraction + write** (source doc, extracted payload, who approved, Epic response) for audit purposes — E&O exposure is real if a bad renewal write goes unnoticed.
5. Once this is validated, promote it out of this shared course repo into its own project with its own dependency set and secrets management.